In [4]:
import sys
from pathlib import Path
import pandas as pd

projectRoot = Path().resolve().parents[0]
print(f"path: {projectRoot}")
sys.path.append(str(projectRoot))

%load_ext autoreload
%autoreload 2

path: /workspace


In [6]:
# Prepare LLM batch inputs
!python /workspace/src/evaluation/benchmark_runner_batch.py prepare \
  /workspace/data/qa/llm_batch/qa_pairs.csv \
  /workspace/data/results/llm_batch/requests.jsonl \
  /workspace/data/results/llm_batch/mapping.csv \
  --provider nebius \
  --model deepseek-ai/DeepSeek-R1-0528 \
  --temperature 0.0 \
  --max_tokens 200

Loaded 1562 QA pairs from /workspace/data/qa/llm_batch/qa_pairs.csv
Using provider: nebius
Prepared 4686 batch requests to /workspace/data/results/llm_batch/requests.jsonl
Wrote mapping to /workspace/data/results/llm_batch/mapping.csv


In [8]:
# Submit LLM batch job and store the output batch ID
from datetime import datetime
import subprocess

# Generate the file path for storing the batch ID
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
batch_id_file = projectRoot / f"BATCH_ID/{timestamp}_batchID"

# Run the command and capture the batch ID

result = subprocess.run(
    [
        "python", "/workspace/src/evaluation/benchmark_runner_batch.py", "submit",
        "/workspace/data/results/llm_batch/requests.jsonl",
        "--provider", "nebius",
        "--window", "24h"
    ],
    stdout=subprocess.PIPE,
    text=True,
    check=True
)

# Extract the batch ID from the command output
batch_id = result.stdout.strip()

# Save the batch ID to the file
batch_id_file.parent.mkdir(parents=True, exist_ok=True)
with open(batch_id_file, "w") as file:
    file.write(batch_id)

print(f"Batch ID saved to: {batch_id_file}")

Batch ID saved to: /workspace/BATCH_ID/20260104_002400_batchID


In [13]:
# Check status of LLM batch job
# batch_id = "<batch_id>"  # Replace with the actual batch ID

# Read batch_id from the file
with open(batch_id_file, "r") as file:
    batch_id = file.read().strip()
batch_id = batch_id.split("id=")[1].split(",")[0]
    
!python /workspace/src/evaluation/benchmark_runner_batch.py status {batch_id} \
    --provider nebius

Batch batch_019b8663-c968-7b9c-b5a8-6d79b3e34c6b status: completed
Counts: BatchRequestCounts(completed=1696, failed=2990, total=4686, prompt_tokens=181006, completion_tokens=339157)
Output file id: 019b8740-981e-7a19-bff6-fdc0e2b10664


In [16]:
# Collect LLM batch results
!python /workspace/src/evaluation/benchmark_runner_batch.py collect \
  {batch_id} \
  /workspace/data/results/llm_batch/output.jsonl \
  /workspace/data/results/llm_batch/mapping.csv \
  /workspace/data/results/llm_batch/deepSeek-R1-0528_results.csv \
  /workspace/data/results/llm_batch/deepSeek-R1-0528_raw.csv \
  --model deepseek-ai/DeepSeek-R1-0528 \
  --provider nebius

Saved batch output JSONL to /workspace/data/results/llm_batch/output.jsonl
Wrote parsed results to /workspace/data/results/llm_batch/deepSeek-R1-0528_results.csv (1562 rows)
Wrote raw outputs to /workspace/data/results/llm_batch/deepSeek-R1-0528_raw.csv (1562 rows)
